# Bronze Layer: ERP Product Category CDC

**Source**: `/Volumes/workspace/bronze/raw_sources/source_erp/PX_CAT_G1V2.csv`  
**Target**: `workspace.bronze.erp_px_cat_g1v2_cdc`  
**Primary Key**: `ID`

**Process:**
1. Read CSV in batch mode
2. Deduplicate on primary key
3. Filter out NULL primary keys
4. **Fast-path**: Skip MERGE if no new data
5. **MERGE**: Update existing, insert new (idempotent)

**Idempotent**: Re-run safe - no duplicates created

## Configuration

In [0]:
# Imports
from pyspark.sql.functions import current_timestamp

# Paths and table names
source_path = "/Volumes/workspace/bronze/raw_sources/source_erp/PX_CAT_G1V2.csv"
bronze_table = "workspace.bronze.erp_px_cat_g1v2_cdc"
primary_key = "ID"

print("✓ Configuration loaded")
print(f"  Source: {source_path}")
print(f"  Target: {bronze_table}")
print(f"  Primary Key: {primary_key}")

## CDC Ingestion with MERGE

In [0]:
# Read CSV file in batch mode
df_raw = (spark.read
  .format("csv")
  .option("header", "true")
  .option("inferSchema", "true")
  .load(source_path)
)

print(f"⌛ Raw records read: {df_raw.count():,}")

# Deduplicate and filter out NULL primary key
df_bronze_batch = (df_raw
  .dropDuplicates([primary_key])
  .filter(f"{primary_key} IS NOT NULL")
  .withColumn("ingestion_timestamp", current_timestamp())
)

print(f"✓ Deduplicated & filtered: {df_bronze_batch.count():,} unique valid records")
print("  Ready for MERGE operation")

In [0]:
# Write to Bronze table using MERGE for deduplication
from delta.tables import DeltaTable

print("Starting batch ingestion with MERGE logic...")

# Check if table exists
table_exists = spark.catalog.tableExists(bronze_table)

if not table_exists:
    # First run: Create table with initial data
    print("  Table doesn't exist - creating with initial load")
    (df_bronze_batch.write
      .format("delta")
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .saveAsTable(bronze_table)
    )
    print(f"\n✓ Initial load complete: {spark.table(bronze_table).count():,} records")
else:
    # Subsequent runs: Check if there's new data before MERGE
    print("  Table exists - checking for new data...")
    
    # Get source and target primary keys
    source_ids = df_bronze_batch.select(primary_key).distinct()
    target_table_df = spark.table(bronze_table)
    target_ids = target_table_df.select(primary_key).distinct()
    
    # Find new IDs in source that don't exist in target
    new_ids = source_ids.join(target_ids, primary_key, "left_anti")
    new_count = new_ids.count()
    
    # Compare counts
    source_count = source_ids.count()
    target_count = target_ids.count()
    
    if new_count == 0 and source_count == target_count:
        print(f"\n⏭️  No new data detected - skipping MERGE")
        print(f"   Source: {source_count:,} records")
        print(f"   Target: {target_count:,} records")
        print(f"   New: 0 records")
        print(f"\n✓ Bronze table unchanged: {target_count:,} total records")
    else:
        # New data detected - run MERGE
        print(f"  ✓ New data detected: {new_count:,} new records")
        print("  Running MERGE...")
        
        # Load target Delta table
        target_table = DeltaTable.forName(spark, bronze_table)
        
        # Perform MERGE operation
        (target_table.alias("target")
          .merge(
              df_bronze_batch.alias("source"),
              f"target.{primary_key} = source.{primary_key}"
          )
          .whenMatchedUpdateAll()
          .whenNotMatchedInsertAll()
          .execute()
        )
        
        print(f"\n✓ MERGE complete: {spark.table(bronze_table).count():,} total records")

## Verification

Check ingestion results

In [0]:
%sql
-- Verify bronze table
SELECT 
  COUNT(*) as total_records,
  COUNT(DISTINCT ID) as unique_categories,
  MIN(ingestion_timestamp) as first_ingestion,
  MAX(ingestion_timestamp) as last_ingestion
FROM bronze.erp_px_cat_g1v2_cdc

In [0]:
%sql
-- Sample records from bronze
SELECT *
FROM bronze.erp_px_cat_g1v2_cdc
ORDER BY ingestion_timestamp DESC
LIMIT 10